# 🌡️ **Thermodynamic Analysis on the Skew-T Log-P Diagram**

---

### **Context**
In this notebook, column-scale atmospheric **thermodynamic** conditions are diagnosed through the analysis of radiosonde soundings on the **Skew-T log-P diagram**. Vertical stability, convective potential, and the kinematic environment of the atmospheric column are quantified through a suite of complementary indices and the vertical wind profile.

### **Learning Goals:**

- 🎯 **Goal 1 | The Skew-T Log-P Background**: Understand the structure of the Skew-T log-P diagram by progressively assembling its five sets of state and process curves
- 🎯 **Goal 2 | Vertical Profile Interpretation**: Interpret the radiosonde profiles on the Skew-T diagram and identify the tropopause and cloud layers
- 🎯 **Goal 3 | Parcel Theory and Convective Indices**: Diagnose convective levels and energy from a surface parcel ascent and assess vertical stability
- 🎯 **Goal 4 | Comprehensive Pre-Storm Assessment**: Extend the convective assessment by synthesizing advanced indices and wind shear diagnostics to comprehensively evaluate the pre-storm potential for storm initiation and organized development

**Run the notebook via the Binder platform:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/weather-analysis-and-forecasting/HEAD?urlpath=notebooks/notebooks/04_skewt_analysis.ipynb)

---

## ⚙️ Setup, Configuration, and Data Acquisition

The three cells below execute the full setup, configuration, and data acquisition pipeline.  

First, the necessary libraries are imported and the environment is configured. 

Then, the target station, analysis date and hour, and diagram extent are configured following the instructions below:

1. **Choose your Data Mode** ('realtime' for the latest observations or 'retrospective' for historical data)
2. **Set your target station** using the WMO station identifier
3. **Set your target date and hour** (only applicable for 'retrospective' mode)
4. **Adjust the diagram extent** to modify the displayed pressure and temperature range

> **Note on data mode:** The default configuration analyzes a high-impact retrospective hail event over Greece. If selecting 'realtime' mode, choose a station located in or near regions of organized synoptic-scale weather development identified from the analysis performed with the previous notebook (03_synoptic_analysis_II.ipynb).

Finally, the radiosonde sounding data are retrieved from [University of Wyoming upper-air database](https://weather.uwyo.edu/upperair/sounding.shtml). 

In [ ]:
# Import required libraries
from __future__ import annotations

import sys
import importlib
from datetime import datetime
import warnings
from pathlib import Path

# Core data science stack
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# Visualization
import matplotlib.pyplot as plt

# Meteorological calculations and Skew-T
import metpy.calc as mpcalc
from metpy.plots import SkewT
from metpy.units import units

# Add local utils to path
sys.path.insert(0, str(Path('../utils').resolve()))

# Import and reload to pick up any changes without kernel restart
from wyoming_raob import fetch_latest_sounding, fetch_retrospective_sounding
import plot_helpers as _ph; importlib.reload(_ph)
from plot_helpers import (
    init_skewt_figure,
    add_skewt_mixing_lines,
    add_skewt_dry_adiabats,
    add_skewt_moist_adiabats,
    plot_skewt_sounding,
    plot_skewt_parcel_diagnosis,
    plot_skewt_cape_cin,
    plot_skewt_convective_levels,
    plot_skewt_instability_panel,
    plot_skewt_winds,
)

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 12),
    'figure.dpi': 100,
    'font.size': 12,
})

print('[OK] All libraries loaded successfully!')

In [ ]:
# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

# ---------------------------------------------------------------------------
# DATA MODE
# ---------------------------------------------------------------------------
# 'realtime'      -> target the most recent completed GFS analysis cycle.
#
# 'retrospective' -> fetch a specific historical analysis cycle defined by
#                    TARGET_DATE and TARGET_HOUR.
DATA_MODE = 'retrospective'   # <-- CHANGE TO 'realtime' FOR LATEST ANALYSIS

# ---------------------------------------------------------------------------
# TARGET STATION
# ---------------------------------------------------------------------------
STATION_ID  = '16716'           # WMO number — LGAT Athens

# ---------------------------------------------------------------------------
# RETROSPECTIVE SETTINGS (used only when DATA_MODE = 'retrospective')
# ---------------------------------------------------------------------------
TARGET_DATE = '2019-10-04'      # UTC date (YYYY-MM-DD); retrospective mode only
TARGET_HOUR = 12                # Sounding hour (0 or 12 UTC); retrospective mode only

# ---------------------------------------------------------------------------
# DIAGRAM SETTINGS
# ---------------------------------------------------------------------------
P_MIN =  100   # hPa — diagram top
P_MAX = 1050   # hPa — diagram bottom
T_MIN =  -40   # °C  — left edge of temperature axis
T_MAX =   40   # °C  — right edge of temperature axis

# ---------------------------------------------------------------------------
# OUTPUT DIRECTORY
# ---------------------------------------------------------------------------
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
if DATA_MODE == 'retrospective':
    print(f'[CONFIG] Retrospective mode — station {STATION_ID}, {TARGET_DATE} {TARGET_HOUR:02d}:00 UTC')
else:
    print(f'[CONFIG] Realtime mode — station {STATION_ID}')

In [ ]:
# ---------------------------------------------------------------------------
# Dispatch: real-time vs retrospective
# ---------------------------------------------------------------------------
if DATA_MODE == 'retrospective':
    df, valid_dt = fetch_retrospective_sounding(STATION_ID, TARGET_DATE, TARGET_HOUR)
else:
    df, valid_dt = fetch_latest_sounding(STATION_ID)

# Validate the returned sounding matches the requested target
assert valid_dt.strftime('%Y-%m-%d') == TARGET_DATE or DATA_MODE == 'realtime', (
    f"Sounding date mismatch: got {valid_dt.strftime('%Y-%m-%d')}, expected {TARGET_DATE}"
)
assert DATA_MODE == 'realtime' or valid_dt.hour == TARGET_HOUR, (
    f"Sounding hour mismatch: got {valid_dt.hour:02d}:00, expected {TARGET_HOUR:02d}:00"
)
assert not df.empty, "Sounding DataFrame is empty — check station ID and date"

print(f'Station : {STATION_ID}')
print(f'Valid   : {valid_dt.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Levels  : {len(df)} pressure levels')
df[['pressure', 'height', 'temperature', 'dewpoint', 'wind_direction', 'wind_speed']].head()

---

# Section 1: The Skew-T Log-P Background

## 1.1 The Five Curve Sets of the Skew-T Log-P Diagram

A thermodynamic diagram is a specialized chart on which radiosonde observations are plotted to characterize the vertical structure of the atmosphere at a given location and time. From a complete diagram, the analyst can directly assess the vertical profile of temperature and humidity, the static stability of the atmosphere, and the potential for thunderstorm development through composite instability indices. Among the various diagrams in use (Emagram, Tephigram, Stüve, Skew-T log-P), the **Skew-T log-P diagram** is the most widely adopted.

Every complete Skew-T log-P diagram is built from five sets of curves, organized into two functional groups:

| Curve Set | Type | Function |
| --- | --- | --- |
| Isobars | State | Levels of constant pressure |
| Isotherms | State | Lines of constant temperature |
| Mixing-ratio lines | State | Lines of constant water vapor content |
| Dry adiabats | Process | Adiabatic temperature change rate of an unsaturated parcel |
| Moist adiabats | Process | Adiabatic temperature change rate a saturated parcel |

## 1.2 Isobars & Isotherms

**Isobars** are drawn as horizontal lines of constant pressure on a logarithmic vertical scale. Spacing increases with height because air density decreases with height. The same pressure interval (e.g., 100 hPa) corresponds to a thicker layer at higher altitudes.

**Isotherms** are drawn as tilted straight lines of constant temperature on a linear scale. The 0°C isotherm is usually highlighted, since it is a critical reference for the precipitation phase (rain, snow, freezing rain).

In [ ]:
fig, skew = init_skewt_figure(P_MIN, P_MAX, T_MIN, T_MAX)
plt.show()

## 1.3 Mixing-Ratio Lines

**Mixing-ratio** lines are drawn as tilted dashed lines of constant water vapor mixing ratio (g kg⁻¹), with a slope shallower than the isotherms.

In [ ]:
fig, skew = init_skewt_figure(P_MIN, P_MAX, T_MIN, T_MAX)
add_skewt_mixing_lines(skew)
plt.show()

## 1.4 Dry Adiabats

**Dry adiabats** are curves representing the dry adiabatic lapse rate $\Gamma_d \approx 9.8$ °C km⁻¹. An unsaturated parcel forced to ascend or descend adiabatically follows these curves. 

In [ ]:
fig, skew = init_skewt_figure(P_MIN, P_MAX, T_MIN, T_MAX)
add_skewt_mixing_lines(skew)
add_skewt_dry_adiabats(skew, T_MIN, T_MAX)
plt.show()

## 1.5 Moist Adiabats

**Moist adiabats** are curves representing the saturated adiabatic lapse rate $\Gamma_s$. This is not constant, as it depends on temperature, but $\Gamma_s$ approaches $\Gamma_d$ at very low temperatures. A saturated parcel forced to ascend or descend follows these curves. 

With this fifth set the background of the Skew-T log-P diagram is complete.

In [ ]:
fig, skew = init_skewt_figure(P_MIN, P_MAX, T_MIN, T_MAX)
add_skewt_mixing_lines(skew)
add_skewt_dry_adiabats(skew, T_MIN, T_MAX)
add_skewt_moist_adiabats(skew, T_MIN, T_MAX)
plt.show()

# Save the Skew-T background plot in outputs directory
out_path = OUTPUT_DIR / f"skewt_background_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.png"
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"[OK] Saved to {out_path}")

---

# Section 2: Vertical Profile Interpretation

The radiosonde sounding provides simultaneous in-situ measurements of pressure, temperature, dew point, and wind from the surface to the lower stratosphere. When plotted on the Skew-T diagram, the temperature and dew-point temperature profiles trace the vertical thermodynamic state of the atmosphere at the launch site. 

Two structural features are diagnosed directly from these thermodynamic profiles. The **tropopause** is identified from the temperature profile as the level where the lapse rate transitions to a persistent isothermal layer or deep temperature inversion, marking the upper boundary of the troposphere. **Cloud layers** are identified as layers in which the dew-point depression ($T - T_d$) falls below approximately 3°C, indicating very humid air or near-saturated air (Relative Humidity ≥ 70%).

The cell below overlays the radiosonde temperature and dew-point profiles for the target station, date, and time onto the complete background diagram produced in Section 1.

In [ ]:
# Construct pint-tagged arrays from the sounding DataFrame
p  = df['pressure'].values    * units.hPa
T  = df['temperature'].values * units.degC
Td = df['dewpoint'].values    * units.degC

# Build the complete background and overlay the observed profiles
fig, skew = plot_skewt_sounding(p, T, Td, P_MIN, P_MAX, T_MIN, T_MAX,
                                station_id=STATION_ID, valid_dt=valid_dt)
plt.show()

# Save the Skew-T sounding plot in outputs directory
out_path = OUTPUT_DIR / f"skewt_sounding_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.png"
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"[OK] Saved to {out_path}")

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.1 — Tropopause and Cloud Layers

**Goal:** Diagnose the upper boundary of the troposphere and regions of likely cloud formation.

Applying the theoretical framework provided previously, evaluate the environmental profiles to:

1. Identify the tropopause from the temperature profile.
2. Locate potential cloud layers.

</div>

---

# Section 3: Parcel Theory and Convective Indices

## 3.1 Parcel Ascent and Convective Levels

The lifting of a near-surface air parcel along the appropriate adiabats traces the **parcel path** and reveals the convective potential of the atmosphere. Initially, the unsaturated air parcel is lifted from the surface, with its temperature cooling parallel to the nearest dry adiabat and its dew point tracking parallel to the nearest line of constant mixing ratio. Where these two paths intersect, the parcel reaches saturation. Beyond this point, the now-saturated parcel continues its ascent by tracing parallel to the nearest moist adiabat for the remainder of its trajectory.

Three key convective levels dictate this vertical trajectory:

- **Lifting Condensation Level (LCL):** The pressure level at which a lifted, unsaturated air parcel cools to its dew point and reaches saturation. The LCL dictates the cloud base height for cumuliform clouds formed by mechanical (forced) lifting.
- **Level of Free Convection (LFC):** The pressure level at which a saturated, mechanically lifted parcel first becomes warmer than the surrounding environment. Above the LFC, the parcel begins to rise freely without an external forcing mechanism.
- **Equilibrium Level (EL):** The pressure level at which the freely rising parcel cools enough to once again equal the environmental temperature, ceasing free convection. The EL marks the typical anvil top of a mature cumulonimbus cloud and frequently lies near or just above the tropopause. 

The cell below computes the surface parcel ascent for the examined sounding and overlays the resulting parcel path onto the Skew-T log-P diagram.

In [ ]:
# Surface parcel ascent
parcel = mpcalc.parcel_profile(p, T[0], Td[0])

# Plot the parcel path along with the observed sounding
fig, skew = plot_skewt_sounding(p, T, Td, P_MIN, P_MAX, T_MIN, T_MAX,
                                station_id=STATION_ID, valid_dt=valid_dt)
plot_skewt_parcel_diagnosis(skew, p, T, Td, parcel)
plt.show()

# Save the Skew-T sounding and parcel plot in outputs directory
out_path = OUTPUT_DIR / f"skewt_sounding_parcel_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.png"
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"[OK] Saved to {out_path}")

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 3.1 — Parcel Ascent and Convective Levels

**Goal:** Diagnose convective levels from a surface parcel ascent.

Following the theoretical framework detailed above, examine the plotted parcel path and the environmental temperature profile on the Skew-T log-P diagram, and perform the following: 

1. Visually identify the approximate pressure altitudes of the three convective levels (LCL, LFC, and EL).
2. Validate your visual estimates by running the code cell below, which computes and overlays the exact levels onto the diagram.


</div>

In [ ]:
# Compute convective levels 
lcl_p, lcl_T = mpcalc.lcl(p[0], T[0], Td[0])
lfc_p, lfc_T = mpcalc.lfc(p, T, Td)
el_p,  el_T  = mpcalc.el(p, T, Td)

# Check for missing LFC or EL and print messages accordingly
if np.isnan(lfc_p.m):
    print('No LFC — stable column')
if np.isnan(el_p.m):
    print('No EL — parcel never reaches equilibrium level')

# Print the convective levels
print(f'LCL  : {lcl_p.m:.1f} hPa')
print(f'LFC  : {"N/A" if np.isnan(lfc_p.m) else f"{lfc_p.m:.1f} hPa"}')
print(f'EL   : {"N/A" if np.isnan(el_p.m)  else f"{el_p.m:.1f} hPa"}')

# Build the full Skew-T with parcel diagnosis and convective levels annotated
fig, skew = plot_skewt_convective_levels(
    p, T, Td, parcel,
    lcl_p=lcl_p, lfc_p=lfc_p, el_p=el_p,
    p_min=P_MIN, p_max=P_MAX, t_min=T_MIN, t_max=T_MAX,
    station_id=STATION_ID, valid_dt=valid_dt,
)
plt.show()

## 3.2 Convective Energy and Vertical Stability

The areas bounded by the parcel path, the environmental temperature profile, and the three key convective levels identified above, quantify the energetic state and vertical stability of the atmospheric column through two indices:

- **Convective Inhibition (CIN):** The energy (J kg⁻¹) required to lift an air parcel from the surface to the LFC, representing a stable layer that suppresses storm development. On the Skew-T log-P diagram, CIN is specifically the area between the LCL and the LFC where the parcel path is cooler than the environmental temperature profile. This stable cap must be overcome by a mechanical lifting mechanism, or eroded by strong surface heating and upper-level cooling, for the parcel to reach free convection. The magnitude of this cap dictates the difficulty of triggering a storm:
    - Greater than 0 J kg⁻¹ (No cap). Allows weak convection.
    - 0 to –20 J kg⁻¹ (Weak): Triggering is easy, allowing for potential air-mass thunderstorms.
    - –20 to –60 J kg⁻¹ (Moderate): Provides the best conditions for severe thunderstorms, as it traps heat and humidity in the boundary layer to serve as storm fuel before a trigger mechanism breaks the cap.
    - –60 to –100 J kg⁻¹ (Strong): Difficult to break, requiring an exceptionally strong lifting mechanism.
    - Lower than –100 J kg⁻¹ (Intense): Thunderstorm triggering is highly unlikely.

- **Convective Available Potential Energy (CAPE):** A primary measure of atmospheric instability, representing the maximum kinetic energy (J kg⁻¹) acquired by an unstable parcel rising freely. On the Skew-T log-P diagram, CAPE is specifically the area between the LFC and the EL where the parcel path is warmer than the environmental temperature profile. The magnitude of CAPE dictates the degree of instability and potential storm severity:
    - 0 to 300 J kg⁻¹: Mostly stable, with little to no thunderstorm activity.
    - 300 to 1000 J kg⁻¹: Marginally unstable, supporting weak thunderstorms.
    - 1000 to 2500 J kg⁻¹: Moderately unstable, where moderate thunderstorms are likely and severe storms are possible.
    - 2500 to 3500 J kg⁻¹: Strongly unstable, making severe thunderstorms and possible tornadoes likely.
    - Greater than 3500 J kg⁻¹: Extremely unstable, indicating that severe thunderstorms and tornadoes are highly likely if triggered.

> **Note on CAPE**: While larger CAPE values dictate a greater potential for severe storm intensity, CAPE alone does not guarantee thunderstorm initiation, as discussed in the following section.

> **Note on thresholds**: The provided thresholds, adopted from Stull (2017), are empirical/statistical, and must be applied with caution in different geographical areas, as local climatology and topography can significantly alter the values required to trigger severe instability.

The cell below calculates the CIN and CAPE and shades their respective areas between the parcel path and the environmental temperature profile on the Skew-T log-P diagram.

In [ ]:
# Compute CAPE and CIN
cape, cin = mpcalc.cape_cin(p, T, Td, parcel)

# Print CAPE and CIN values
print(f'CIN  : {cin.m:.1f} J/kg')
print(f'CAPE : {cape.m:.1f} J/kg')

# Build diagram: sounding + parcel path + convective levels
fig, skew = plot_skewt_convective_levels(
    p, T, Td, parcel,
    lcl_p=lcl_p, lfc_p=lfc_p, el_p=el_p,
    p_min=P_MIN, p_max=P_MAX, t_min=T_MIN, t_max=T_MAX,
    station_id=STATION_ID, valid_dt=valid_dt,
)
# Shade CAPE/CIN areas and annotate values
plot_skewt_cape_cin(skew, p, T, parcel,
                    lcl_p=lcl_p, el_p=el_p,
                    cape=cape, cin=cin,
                    fig=fig, output_dir=OUTPUT_DIR)
plt.show()

<div style="background-color: #d1e7dd; padding: 15px; border-radius: 8px; border-left: 4px solid #198754e0;">

## Checkpoint: Evaluating Thunderstorm Potential

Using the CAPE and CIN values computed above, evaluate the strength of the capping inversion, the degree of vertical instability, and the overall thunderstorm potential for the examined atmospheric profile.

<div>

---

# Section 4: Comprehensive Pre-Storm Assessment

## 4.1 Pre-storm conditions 

The analysis in Section 3 quantified two of the environmental conditions governing convective storm development: the capping inversion, which must be overcome for storm initiation, and CAPE, which drives the updraft once the parcel reaches free convection. Thunderstorm initiation and behavior, however, depends on the simultaneous presence of three conditions for initiation and a fourth that determines longevity and organization.

**Conditions for Storm Initiation:**

- **Trigger mechanism:** An external forcing, such as a synoptic front, orographic lifting, or large-scale ascent, must lift boundary-layer air parcels through the stable cap to their LFC. CIN quantifies the amount of energy the trigger needs to break through this stable layer. 
- **Conditional instability:** A deep layer aloft where lifted, saturated air parcels become warmer than their surrounding environment, must be present. CAPE quantifies the magnitude of this available instability and energy.
- **Boundary-layer moisture:** High water vapor content in the atmospheric boundary layer provides the latent heat fuel to sustain storm updrafts.

**Condition for Storm Organization and Longevity:**

- **Vertical wind shear:** Strong shear is necessary to make storms last longer and become severe. It separates updraft and downdraft regions, enabling organized multicellular and supercellular structures.

The convective environment should therefore be evaluated through a suite of complementary diagnostics, each targeted at one or more of these conditions.

## 4.1 MUCAPE and the K-Index

Surface-based CAPE is representative primarily when the surface layer is actively coupled to the convective boundary layer, as seen in afternoon soundings. However, when the surface is cold and stable, such as in early-morning soundings, nocturnal environments, or instances of elevated convection above a surface inversion, surface-based calculations can significantly underestimate the available atmospheric instability.

**Most Unstable CAPE (MUCAPE)** extends the atmospheric instability assessment beyond the surface layer. It calculates many different CAPEs for air parcels that start from every height in the bottom 300 hPa of the pre-storm environmental sounding, and then selects the one that gives the greatest value. 

The **K-index** addresses the boundary-layer moisture factor. It combines lower- and mid-troposphere temperature and dew point into a single index:

$$K = (T_{850} - T_{500}) + T_{d,850} - (T_{700} - T_{d,700})$$

with its values values corresponding to the following probabilities of thunderstorm occurrence:

- Lower than 15: 0% probability
- 15 to 20: < 20% probability
- 21 to 25: 20% to 40% probability
- 26 to 30: 40% to 60% probability
- 31 to 35: 60% to 80% probability
- 36 to 40: 80% to 90% probability
- Greater than 40: > 90% probability

The cell below computes MUCAPE and the K-index and displays them alongside the
surface-parcel CAPE and CIN from Section 3.

In [ ]:
# Compute MUCAPE and K-index 
mu_cape, mu_cin = mpcalc.most_unstable_cape_cin(p, T, Td, depth=300 * units.hPa)
k = mpcalc.k_index(p, T, Td)

# Print the indices
print(f'MUCAPE              : {mu_cape.m:.1f} J/kg')
print(f'K-index             : {k.m:.1f} °C')

# Final diagram
fig, skew = plot_skewt_instability_panel(
    p, T, Td, parcel,
    lcl_p=lcl_p, lfc_p=lfc_p, el_p=el_p,
    mu_cape=mu_cape, k=k,
    cape=cape, cin=cin,
    p_min=P_MIN, p_max=P_MAX, t_min=T_MIN, t_max=T_MAX,
    station_id=STATION_ID, valid_dt=valid_dt,
)
plt.show()

## 4.2 Vertical Wind Profile  

Wind shear is assessed through the vertical wind profile, plotted as wind barbs to the right of the Skew-T diagram. Two diagnostically distinct aspects of the profile are examined: 

- **Directional shear** describes changes in wind direction with height. Veering (clockwise rotation with height) indicates warm air advection (WAA) in the lower troposphere and strengthens the storm environment, whereas backing (counterclockwise rotation) indicates cold air advection (CAA) and weakens it

- **Speed shear** across the 0–6 km layer quantifies the deep-layer kinematic environment. Strong vertical shear forces the storm updraft to tilt, placing the precipitation downdraft to one side of the updraft rather than directly beneath it. This physical separation prevents the downdraft from undercutting the updraft, prolongs the storm's lifetime and enables supercellular organization

> The indices computed in this notebook represent a subset of the diagnostics in operational use. Among those frequently applied alongside or in place of the measures above are the Lifted Index (LI), the Total Totals Index (TT), the Energy-Helicity Index (EHI), and the Supercell Composite Parameter (SCP).

The cell below plots the wind barb profile on the full Skew-T diagram and computes the 0–6 km bulk shear magnitude.

In [ ]:
# Wind and height arrays 
u   = df['u_wind'].values * units('knots')
v   = df['v_wind'].values * units('knots')
hgt = df['height'].values * units.m

# Compute 0–6 km bulk wind shear
u_shear, v_shear = mpcalc.bulk_shear(p, u, v, height=hgt, depth=6 * units.km)
shear_mag = np.hypot(u_shear.m, v_shear.m)

# Final diagram with wind profile analysis
fig, skew = plot_skewt_winds(
    p, T, Td, parcel, u, v,
    lcl_p=lcl_p, lfc_p=lfc_p, el_p=el_p,
    mu_cape=mu_cape, k=k,
    shear_mag=shear_mag,
    cape=cape, cin=cin,
    p_min=P_MIN, p_max=P_MAX, t_min=T_MIN, t_max=T_MAX,
    station_id=STATION_ID, valid_dt=valid_dt,
)
print(f'0–6 km bulk shear : {shear_mag:.1f} kt')
plt.show()

<div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">

📝 **Task 4.1 — Convective Environment Synthesis**

**Goal**: Comprehensively evaluate the pre-storm convective environment by extending the Section 3 diagnostics through MUCAPE, the K-index, and wind shear.

Using the values computed above:
1. Compare surface-based CAPE and MUCAPE.
2. Interpret the K-index against its conventional thresholds and evaluate the 0–6 km bulk shear for storm organization potential.
3. Integrating with the CIN assessment from Section 3, do the combined diagnostics support storm initiation and organized development? 
4. Validate your pre-storm assessment against the environmental characterization reported in [Papavasileiou et al. (2022)](https://doi.org/10.1016/j.atmosres.2022.106341).

</div>

<div style="background-color: #f8d7da; border-left: 4px solid #dc3545; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">

🔴 **Pro Task — Connecting to Realtime Synoptic Diagnosis**

**Goal:** Apply the full Skew-T analysis workflow to a synoptically active scenario and connect the column-scale thermodynamic diagnosis to the synoptic context established in the previous notebook.

Run `03_synoptic_analysis_II.ipynb` in realtime mode and  identify any large-scale environment favorable to cloud formation and precipitation. Switch `DATA_MODE` to `'realtime'`, set `STATION_ID` to the WMO number of the nearest radiosonde station within that region, and re-run all cells. Compare the column-scale thermodynamic analysis with the synoptic-scale diagnosis from `03_synoptic_analysis_II.ipynb`.

> To identify the nearest sounding station, consult the interactive map at the [University of Wyoming upper-air database](https://weather.uwyo.edu/upperair/sounding.shtml).


You can also explore other dates of interest from your own research.

</div>

## 🏁 Summary & Key Takeaways

In this notebook, column-scale thermodynamic conditions were diagnosed through the analysis of radiosonde soundings on the Skew-T log-P diagram. Vertical stability, convective potential, and the kinematic environment of the atmospheric column were assessed through parcel theory, instability indices, and the vertical wind profile.

**Key Takeaways:**

* ✅ **Skew-T Log-P Background:** Five sets of state and process curves (isobars, isotherms, mixing-ratio lines, dry adiabats, and moist adiabats) constitute the complete background diagram.
* ✅ **Vertical Profile Interpretation:** The analysis of temperature and dew-point profiles reveal the tropopause and identify potential cloud layers. 
* ✅ **Parcel Theory and Convective Levels:** LCL, LFC, and EL define the convective structure of the atmospheric column and delimit the areas of CIN and CAPE on the Skew-T diagram.
* ✅ **Comprehensive Pre-Storm Assessment:** Composite diagnostics, such as MUCAPE, the K-index, and 0–6 km bulk shear, extend the convective assessment enabling a comprehensive pre-storm evaluation of storm initiation and organization potential

---

## References
1. Stull, R. (2017). *Practical Meteorology: An Algebra-based Survey of Atmospheric Science.* University of British Columbia. Available at: [https://www.eoas.ubc.ca/books/Practical_Meteorology](https://www.eoas.ubc.ca/books/Practical_Meteorology)
2. Milrad, S. (2018). *Synoptic Analysis and Forecasting: An Introductory Toolkit.* Elsevier. https://doi.org/10.1016/C2015-0-05604-0 
3. Papavasileiou, G. (2022). *Observational and numerical study of a giant hailstorm in Attica, Greece, on 4 October 2019.* Atmospheric Research. https://doi.org/10.1016/j.atmosres.2022.106341 


---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>